# 02 — Scraper manual de detalle (Bronce)

Visita individualmente cada aviso pendiente (los que ya están en la tabla
`avisos` pero todavía no tienen fila en `avisos_detalle`) y extrae su
descripción completa, características, coordenadas y puntos de interés
cercanos — tal como los devuelve el sitio, en texto crudo, sin transformar.

**Alcance de este notebook:**
- Corrida **manual**, sin scheduling, se ejecuta a mano cuando se quiera
  cargar detalle nuevo.
- Extracción vía regex sobre el texto visible de la página, más un bloque
  JSON embebido para resolver los puntos de interés cercanos (colegios,
  paraderos, supermercados, etc.).
- Todo el scraping ocurre en **pandas puro** (sin la API de DataFrames de
  Spark); Spark se usa únicamente para leer los pendientes (`LEFT JOIN`) y
  para el `MERGE INTO` final hacia la tabla Delta.
- Se detiene de inmediato ante un CAPTCHA, dejando guardado en memoria lo ya
  procesado hasta ese punto.

**Qué NO hace este notebook (queda para etapas posteriores de limpieza):**
- No aplica la lógica completa de `gastos_comunes` (casos de placeholder,
  outliers, miles reales), solo saca el separador de miles para que el dato
  quepa en una columna numérica.
- No calcula distancias (Haversine) a puntos de referencia, se calculan
  más adelante a partir de `latitud`/`longitud`.
- No imputa superficie ni antigüedad, ni resuelve columnas derivadas de
  fuentes externas.

**Requisito previo:** debe existir al menos un aviso cargado en la tabla
`avisos` antes de correr este notebook, para que haya algo sobre lo cual
buscar detalle pendiente.

### 1. Configuración y constantes del scraper
Comunas, regex de extracción, selectores CSS, y coordenadas de referencia, misma lógica que el scraper de investigación original, sin transformar datos.

In [0]:
import re
import json
import time
import random
import logging
from datetime import date, timedelta
from math import radians, sin, cos, sqrt, atan2
from typing import Optional

import requests
import pandas as pd
from bs4 import BeautifulSoup

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

DELAY_MIN = 2.0
DELAY_MAX = 4.0
TIMEOUT_REQUEST_SEG = 15
REINTENTOS_TRAS_ERROR = 2
BACKOFF_REINTENTO_MIN = 3.0
BACKOFF_REINTENTO_MAX = 6.0

BASE_URL = "https://www.portalinmobiliario.com"
OPERACION = "arriendo"

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
)

HEADERS_BASE = {
    "User-Agent": USER_AGENT,
    "Accept-Language": "es-CL,es;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
}


def headers_requests(referer: str) -> dict:
    return {**HEADERS_BASE, "Referer": referer}


RE_FECHA_PUBLICACION = re.compile(r"Publicado (hoy|esta semana|hace [^\n\|]+)", re.IGNORECASE)
RE_SUPERFICIE_TOTAL = re.compile(r"Superficie total\s*([\d.,]+)\s*m", re.IGNORECASE)
RE_SUPERFICIE_UTIL = re.compile(r"Superficie útil\s*([\d.,]+)\s*m", re.IGNORECASE)
RE_DORMITORIOS = re.compile(r"Dormitorios\s*(\d+)")
RE_BANOS = re.compile(r"Baños\s*(\d+)")
RE_ESTACIONAMIENTOS = re.compile(r"Estacionamientos:?\s*(\d+)", re.IGNORECASE)
RE_ANTIGUEDAD = re.compile(r"Antigüedad\s*(\d+)\s*años?", re.IGNORECASE)
RE_AMOBLADO = re.compile(r"Amoblado:?\s*(Sí|No)", re.IGNORECASE)
RE_ADMITE_MASCOTAS = re.compile(r"Admite mascotas:?\s*(Sí|No)", re.IGNORECASE)
RE_CONDOMINIO_CERRADO = re.compile(r"En condominio cerrado:?\s*(Sí|No)", re.IGNORECASE)
RE_BODEGAS = re.compile(r"Bodegas\s*(\d+)", re.IGNORECASE)
RE_GASTOS_COMUNES = re.compile(r"Gastos comunes:?\s*\$?\s*([\d.,]+)", re.IGNORECASE)
RE_GASTOS_COMUNES_RESUMEN = re.compile(r"Gastos comunes\s+desde\s*\$?\s*([\d.,]+)", re.IGNORECASE)
RE_ESTACIONAMIENTO_VISITAS = re.compile(r"Estacionamiento de visitas:?\s*(Sí|No)", re.IGNORECASE)
RE_SOLO_FAMILIAS = re.compile(r"Solo familias:?\s*(Sí|No)", re.IGNORECASE)
RE_MAX_HABITANTES = re.compile(r"Cantidad máxima de habitantes\s*(\d+)", re.IGNORECASE)
RE_PISCINA = re.compile(r"Piscina:?\s*(Sí|No)", re.IGNORECASE)
RE_QUINCHO = re.compile(r"Quincho\D{0,15}?:?\s*(Sí|No)", re.IGNORECASE)
RE_CONSERJERIA = re.compile(r"Conserjería:?\s*(Sí|No)", re.IGNORECASE)
RE_ASCENSOR = re.compile(r"Ascensor:?\s*(Sí|No)", re.IGNORECASE)
RE_PISO_UNIDAD = re.compile(r"Número de piso de la unidad\s*(\d+)", re.IGNORECASE)
RE_DEPTOS_POR_PISO = re.compile(r"Departamentos por piso\s*(\d+)", re.IGNORECASE)
RE_LATITUD = re.compile(r'"latitude":"(-?[\d.]+)"')
RE_LONGITUD = re.compile(r'"longitude":"(-?[\d.]+)"')
RE_LATLON_MAPA = re.compile(r"center=(-?[\d.]+)%2C(-?[\d.]+)")

SELECTORES_DESCRIPCION = [
    "[data-testid='core-description'] p",
    "p.ui-pdp-description__content",
    "div.ui-pdp-description",
]

RADIO_MAXIMO_POI_M = 500

SUBCATEGORIAS_POI = {
    "paraderos": "paraderos",
    "estaciones de metro": "estaciones_metro",
    "jardines infantiles": "jardines_infantiles",
    "colegios": "colegios",
    "universidades": "universidades",
    "plazas": "plazas",
    "supermercados": "supermercados",
    "farmacias": "farmacias",
    "centros comerciales": "centros_comerciales",
    "hospitales": "hospitales",
    "clinicas": "clinicas",
}

COMUNA_CENTROS = {
    "concepcion-biobio":          (-36.8265, -73.0524),
    "talcahuano-biobio":          (-36.7249, -73.1149),
    "hualpen-biobio":             (-36.7690, -73.1000),
    "san-pedro-de-la-paz-biobio": (-36.8380, -73.0970),
    "chiguayante-biobio":         (-36.9280, -73.0230),
    "penco-biobio":               (-36.7420, -72.9970),
    "tome-biobio":                (-36.6180, -72.9570),
    "coronel-biobio":             (-37.0270, -73.1370),
    "hualqui-biobio":             (-36.9670, -72.9420),
    "lota-biobio":                (-37.0920, -73.1600),
}
CENTRO_CONCEPCION = COMUNA_CENTROS["concepcion-biobio"]

DIAS_ESTA_SEMANA = 3

### 2. Funciones de extracción y parsing
Adaptador HTML (simula la interfaz de Playwright sobre HTML estático) + funciones que extraen cada campo del detalle del aviso (descripción, coordenadas, puntos de interés, fecha de publicación, etc.). Todo se guarda como texto crudo, sin limpiar.

In [0]:
class _LocatorHTMLEstatico:
    def __init__(self, soup, selector):
        self._soup = soup
        self._selector = selector

    @property
    def first(self):
        return self

    def count(self):
        if self._selector == "body":
            return 1
        return len(self._soup.select(self._selector))

    def inner_text(self):
        if self._selector == "body":
            nodo = self._soup.body or self._soup
        else:
            elementos = self._soup.select(self._selector)
            if not elementos:
                return ""
            nodo = elementos[0]
        return nodo.get_text("\n", strip=True)


class PaginaHTMLEstatico:
    def __init__(self, html):
        self._html = html or ""
        self._soup = BeautifulSoup(self._html, "lxml")

    def content(self):
        return self._html

    def locator(self, selector):
        return _LocatorHTMLEstatico(self._soup, selector)


def parsear_fecha_relativa(texto):
    if not texto:
        return None
    texto_normalizado = texto.strip().lower()
    if texto_normalizado == "hoy":
        return date.today().isoformat()
    if texto_normalizado == "esta semana":
        return (date.today() - timedelta(days=DIAS_ESTA_SEMANA)).isoformat()

    m = re.search(r"(\d+)\s*(día|dias|semana|mes|meses|año|años)", texto, re.IGNORECASE)
    if not m:
        return None
    cantidad = int(m.group(1))
    unidad = m.group(2).lower()

    if "día" in unidad or "dia" in unidad:
        delta = timedelta(days=cantidad)
    elif "semana" in unidad:
        delta = timedelta(weeks=cantidad)
    elif "mes" in unidad:
        delta = timedelta(days=30 * cantidad)
    elif "año" in unidad:
        delta = timedelta(days=365 * cantidad)
    else:
        return None

    return (date.today() - delta).isoformat()


def determinar_precision_fecha(texto):
    if not texto:
        return None
    if texto.strip().lower() == "esta semana":
        return "aproximada_categoria"
    return "exacta"


def extraer_descripcion(page):
    for selector in SELECTORES_DESCRIPCION:
        loc = page.locator(selector)
        if loc.count() > 0:
            try:
                return loc.first.inner_text().strip()
            except Exception:
                continue
    return None


def extraer_coordenadas(page):
    html_completo = page.content()
    m_lat = RE_LATITUD.search(html_completo)
    m_lon = RE_LONGITUD.search(html_completo)
    if m_lat and m_lon:
        return {"latitud": m_lat.group(1), "longitud": m_lon.group(1)}
    m_mapa = RE_LATLON_MAPA.search(html_completo)
    if m_mapa:
        return {"latitud": m_mapa.group(1), "longitud": m_mapa.group(2)}
    return {"latitud": None, "longitud": None}


def normalizar_texto(texto):
    reemplazos = {"á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u", "ñ": "n"}
    texto = texto.lower().strip()
    for con_tilde, sin_tilde in reemplazos.items():
        texto = texto.replace(con_tilde, sin_tilde)
    return texto


def parsear_distancia_metros(texto):
    m = re.search(r"([\d.,]+)\s*(metros|km)", texto, re.IGNORECASE)
    if not m:
        return None
    valor = float(m.group(1).replace(".", "").replace(",", "."))
    unidad = m.group(2).lower()
    return valor * 1000 if unidad == "km" else valor


def extraer_json_estado_pagina(page):
    html_completo = page.content()
    m = re.search(r"_n\.ctx\.r=(\{.*?\});_n\.ctx\.r\.assets\.manifest=", html_completo, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(1))
    except json.JSONDecodeError:
        return None


def _buscar_categorias_poi(nodo):
    if isinstance(nodo, dict):
        categorias = nodo.get("categories")
        if isinstance(categorias, list) and categorias and isinstance(categorias[0], dict) \
                and "subcategories" in categorias[0]:
            return categorias
        for valor in nodo.values():
            resultado = _buscar_categorias_poi(valor)
            if resultado:
                return resultado
    elif isinstance(nodo, list):
        for item in nodo:
            resultado = _buscar_categorias_poi(item)
            if resultado:
                return resultado
    return None


def _buscar_valor_por_clave(nodo, clave):
    encontrados = []

    def _recorrer(n):
        if isinstance(n, dict):
            if clave in n:
                encontrados.append(n[clave])
            for v in n.values():
                _recorrer(v)
        elif isinstance(n, list):
            for item in n:
                _recorrer(item)

    _recorrer(nodo)
    for valor in encontrados:
        if valor:
            return valor
    return None


def extraer_puntos_interes(estado):
    resultado = {}
    for clave in SUBCATEGORIAS_POI.values():
        resultado[f"cantidad_{clave}"] = None
        resultado[f"distancia_min_m_{clave}"] = None

    if not estado:
        return resultado

    categorias = _buscar_categorias_poi(estado)
    if not categorias:
        return resultado

    for categoria in categorias:
        for subcategoria in categoria.get("subcategories", []):
            nombre = normalizar_texto(subcategoria.get("title", {}).get("text", ""))
            clave = SUBCATEGORIAS_POI.get(nombre)
            if not clave:
                continue

            items = subcategoria.get("items", [])
            distancias_dentro_del_radio = []
            for item in items:
                texto_subtitulo = item.get("subtitle", {}).get("label", {}).get("text", "")
                distancia = parsear_distancia_metros(texto_subtitulo)
                if distancia is not None and distancia <= RADIO_MAXIMO_POI_M:
                    distancias_dentro_del_radio.append(distancia)

            resultado[f"cantidad_{clave}"] = len(distancias_dentro_del_radio)
            resultado[f"distancia_min_m_{clave}"] = (
                min(distancias_dentro_del_radio) if distancias_dentro_del_radio else None
            )

    return resultado


def extraer_detalle(page):
    texto_completo = page.locator("body").inner_text()

    def buscar(patron):
        m = patron.search(texto_completo)
        return m.group(1).strip() if m else None

    gastos_comunes = buscar(RE_GASTOS_COMUNES)
    if gastos_comunes is None:
        gastos_comunes = buscar(RE_GASTOS_COMUNES_RESUMEN)

    fecha_texto = buscar(RE_FECHA_PUBLICACION)
    if fecha_texto and fecha_texto.lower().startswith("hace "):
        fecha_texto = fecha_texto[len("hace "):]

    coordenadas = extraer_coordenadas(page)
    estado = extraer_json_estado_pagina(page)
    puntos_interes = extraer_puntos_interes(estado)
    barrio = _buscar_valor_por_clave(estado, "neighborhood") if estado else None

    return {
        "descripcion": extraer_descripcion(page),
        "fecha_publicacion_texto": fecha_texto,
        "fecha_publicacion_aprox": parsear_fecha_relativa(fecha_texto),
        "fecha_publicacion_precision": determinar_precision_fecha(fecha_texto),
        "superficie_total_m2": buscar(RE_SUPERFICIE_TOTAL),
        "superficie_util_m2": buscar(RE_SUPERFICIE_UTIL),
        "dormitorios": buscar(RE_DORMITORIOS),
        "banos": buscar(RE_BANOS),
        "estacionamientos": buscar(RE_ESTACIONAMIENTOS),
        "antiguedad_anos": buscar(RE_ANTIGUEDAD),
        "amoblado": buscar(RE_AMOBLADO),
        "admite_mascotas": buscar(RE_ADMITE_MASCOTAS),
        "condominio_cerrado": buscar(RE_CONDOMINIO_CERRADO),
        "bodegas": buscar(RE_BODEGAS),
        "gastos_comunes": gastos_comunes,
        "estacionamiento_visitas": buscar(RE_ESTACIONAMIENTO_VISITAS),
        "solo_familias": buscar(RE_SOLO_FAMILIAS),
        "max_habitantes": buscar(RE_MAX_HABITANTES),
        "piscina": buscar(RE_PISCINA),
        "quincho": buscar(RE_QUINCHO),
        "conserjeria": buscar(RE_CONSERJERIA),
        "ascensor": buscar(RE_ASCENSOR),
        "piso_unidad": buscar(RE_PISO_UNIDAD),
        "deptos_por_piso": buscar(RE_DEPTOS_POR_PISO),
        "latitud": coordenadas["latitud"],
        "longitud": coordenadas["longitud"],
        "barrio": barrio,
        **puntos_interes,
    }


def hay_captcha(page):
    contenido = page.content().lower()
    if "captcha" not in contenido[:8000]:
        return False
    texto = page.locator("body").inner_text()
    parece_contenido_real = bool(
        RE_SUPERFICIE_TOTAL.search(texto)
        or RE_SUPERFICIE_UTIL.search(texto)
        or RE_DORMITORIOS.search(texto)
    )
    return not parece_contenido_real


def construir_referer(comuna, tipo_propiedad):
    return f"{BASE_URL}/{OPERACION}/{tipo_propiedad}/{comuna}"


def obtener_detalle_aviso(url, comuna, tipo_propiedad):
    referer = construir_referer(comuna, tipo_propiedad)
    intentos_totales = 1 + REINTENTOS_TRAS_ERROR
    ultimo_status = None
    ultimo_motivo = None

    for intento in range(1, intentos_totales + 1):
        try:
            resp = requests.get(url, headers=headers_requests(referer), timeout=TIMEOUT_REQUEST_SEG)
            ultimo_status = resp.status_code

            if resp.status_code != 200:
                ultimo_motivo = f"status HTTP {resp.status_code}"
                raise ValueError(ultimo_motivo)

            pagina = PaginaHTMLEstatico(resp.text)

            if hay_captcha(pagina):
                return {"resultado": "captcha", "status_http": ultimo_status, "motivo": "captcha"}

            datos = extraer_detalle(pagina)
            return {"resultado": "ok", "datos": datos, "status_http": ultimo_status, "motivo": None}

        except (requests.RequestException, ValueError) as e:
            ultimo_motivo = ultimo_motivo or str(e)
            if intento < intentos_totales:
                espera = random.uniform(BACKOFF_REINTENTO_MIN, BACKOFF_REINTENTO_MAX)
                log.warning(f"Intento {intento}/{intentos_totales} falló para {url} ({ultimo_motivo}). "
                            f"Reintentando en {espera:.1f}s...")
                time.sleep(espera)
            else:
                log.warning(f"Agotados los {intentos_totales} intentos para {url} ({ultimo_motivo}).")

    return {"resultado": "error", "status_http": ultimo_status, "motivo": ultimo_motivo}

### 3. Identificar avisos pendientes de detalle
LEFT JOIN entre `avisos` y `avisos_detalle` en Bronce: trae solo los avisos que todavía no tienen su detalle scrapeado.

In [0]:
pendientes_rows = spark.sql("""
    SELECT a.id_aviso, a.url, a.comuna, a.tipo_propiedad
    FROM gran_concepcion.01_bronce.avisos a
    LEFT JOIN gran_concepcion.01_bronce.avisos_detalle d ON a.id_aviso = d.id_aviso
    WHERE d.id_aviso IS NULL
      AND a.url IS NOT NULL
""").collect()

pendientes = [row.asDict() for row in pendientes_rows]
log.info(f"{len(pendientes)} avisos pendientes de detalle.")

### 4. Scraping de detalle, en memoria
Visita cada aviso pendiente, uno por uno, con delay aleatorio entre requests. Se detiene de inmediato si detecta CAPTCHA, dejando guardado en memoria lo ya procesado.

In [0]:
detalles_nuevos = []
procesados = 0
detenido_por_captcha = False
hoy = date.today().isoformat()

for fila in pendientes:
    id_aviso = fila["id_aviso"]
    url = fila["url"]
    log.info(f"Visitando {id_aviso}: {url}")

    resultado = obtener_detalle_aviso(url, fila["comuna"], fila["tipo_propiedad"])

    if resultado["resultado"] == "captcha":
        log.error(f"CAPTCHA detectado en {url}. Deteniendo la corrida. "
                  f"Lo ya procesado ({procesados} avisos) queda guardado.")
        detenido_por_captcha = True
        break

    if resultado["resultado"] == "error":
        log.warning(f"No se pudo obtener {id_aviso} tras reintentos ({resultado['motivo']}). Se salta.")
        time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
        continue

    datos = dict(resultado["datos"])
    datos["id_aviso"] = id_aviso
    datos["url"] = url
    datos["fecha_scrapeo"] = hoy
    detalles_nuevos.append(datos)
    procesados += 1

    log.info(f"  -> Guardado ({procesados}/{len(pendientes)})")
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

log.info(f"Corrida completa. Procesados: {procesados}. Detenido por CAPTCHA: {detenido_por_captcha}")

### 5. Armar DataFrame y aplicar conversión mínima
Solo se convierten los campos que quedarían mal si se insertaran como texto crudo en columnas numéricas ya existentes: booleanos (Sí/No → 1/0) y gastos_comunes (separador de miles chileno). El resto de la limpieza queda para la capa Plata.

In [0]:
df_detalle = pd.DataFrame(detalles_nuevos)

CAMPOS_BOOLEANOS = [
    "amoblado", "admite_mascotas", "condominio_cerrado", "estacionamiento_visitas",
    "piscina", "quincho", "conserjeria", "ascensor",
]

def _a_booleano(valor):
    if valor is None:
        return None
    texto = str(valor).strip().lower()
    if texto in ("sí", "si"):
        return 1
    if texto == "no":
        return 0
    return None

for campo in CAMPOS_BOOLEANOS:
    if campo in df_detalle.columns:
        df_detalle[campo] = df_detalle[campo].map(_a_booleano)

def _gastos_comunes_a_numero(valor):
    # Solo saca el separador de miles chileno, SIN la lógica completa de
    # 3 casos (placeholder "incluido", outliers) - eso queda para Plata.
    if valor is None:
        return None
    texto = str(valor).strip()
    if not texto:
        return None
    try:
        if "." in texto:
            return float(texto.replace(".", "").replace(",", "."))
        return float(texto.replace(",", "."))
    except (TypeError, ValueError):
        return None

if "gastos_comunes" in df_detalle.columns:
    df_detalle["gastos_comunes"] = df_detalle["gastos_comunes"].map(_gastos_comunes_a_numero)

# NUEVO: superficie_total_m2 / superficie_util_m2 pueden venir con coma
# decimal (ej. "43,69") en vez de punto - CAST de SQL solo acepta punto.
CAMPOS_SUPERFICIE = ["superficie_total_m2", "superficie_util_m2"]

def _superficie_a_numero(valor):
    if valor is None:
        return None
    texto = str(valor).strip()
    if not texto:
        return None
    try:
        return float(texto.replace(",", "."))
    except (TypeError, ValueError):
        return None

for campo in CAMPOS_SUPERFICIE:
    if campo in df_detalle.columns:
        df_detalle[campo] = df_detalle[campo].map(_superficie_a_numero)


print(f"{len(df_detalle)} avisos con detalle nuevo")
df_detalle.head()

### 6. Crear vista temporal para el INSERT
Punto único de contacto con Spark: convierte el DataFrame de pandas en una vista SQL temporal.

In [0]:
spark.createDataFrame(df_detalle).createOrReplaceTempView("detalle_nuevos_tmp")

### 7. MERGE final hacia Bronce
Inserta en `avisos_detalle` solo los avisos que todavía no existían (columnas explícitas + casts, para evitar pérdidas silenciosas de tipo).

In [0]:
%sql
MERGE INTO gran_concepcion.01_bronce.avisos_detalle AS detalle
USING detalle_nuevos_tmp AS nuevos
ON detalle.id_aviso = nuevos.id_aviso
WHEN NOT MATCHED THEN INSERT (
    id_aviso, descripcion, fecha_publicacion_texto, fecha_publicacion_aprox,
    fecha_publicacion_precision, superficie_total_m2, superficie_util_m2,
    dormitorios, banos, estacionamientos, antiguedad_anos,
    amoblado, admite_mascotas, condominio_cerrado, bodegas, gastos_comunes,
    estacionamiento_visitas, solo_familias, max_habitantes, piscina, quincho,
    conserjeria, ascensor, piso_unidad, deptos_por_piso, barrio,
    latitud, longitud,
    cantidad_paraderos, distancia_min_m_paraderos,
    cantidad_estaciones_metro, distancia_min_m_estaciones_metro,
    cantidad_jardines_infantiles, distancia_min_m_jardines_infantiles,
    cantidad_colegios, distancia_min_m_colegios,
    cantidad_universidades, distancia_min_m_universidades,
    cantidad_plazas, distancia_min_m_plazas,
    cantidad_supermercados, distancia_min_m_supermercados,
    cantidad_farmacias, distancia_min_m_farmacias,
    cantidad_centros_comerciales, distancia_min_m_centros_comerciales,
    cantidad_hospitales, distancia_min_m_hospitales,
    cantidad_clinicas, distancia_min_m_clinicas,
    fecha_scrapeo
) VALUES (
    nuevos.id_aviso, nuevos.descripcion, nuevos.fecha_publicacion_texto,
    CAST(nuevos.fecha_publicacion_aprox AS DATE),
    nuevos.fecha_publicacion_precision,
    CAST(nuevos.superficie_total_m2 AS DOUBLE),
    CAST(nuevos.superficie_util_m2 AS DOUBLE),
    CAST(nuevos.dormitorios AS DOUBLE),
    CAST(nuevos.banos AS DOUBLE),
    CAST(nuevos.estacionamientos AS DOUBLE),
    CAST(nuevos.antiguedad_anos AS DOUBLE),
    CAST(nuevos.amoblado AS DOUBLE),
    CAST(nuevos.admite_mascotas AS DOUBLE),
    CAST(nuevos.condominio_cerrado AS DOUBLE),
    CAST(nuevos.bodegas AS DOUBLE),
    CAST(nuevos.gastos_comunes AS DOUBLE),
    CAST(nuevos.estacionamiento_visitas AS DOUBLE),
    nuevos.solo_familias,
    CAST(nuevos.max_habitantes AS DOUBLE),
    CAST(nuevos.piscina AS DOUBLE),
    CAST(nuevos.quincho AS DOUBLE),
    CAST(nuevos.conserjeria AS DOUBLE),
    CAST(nuevos.ascensor AS DOUBLE),
    CAST(nuevos.piso_unidad AS DOUBLE),
    CAST(nuevos.deptos_por_piso AS DOUBLE),
    nuevos.barrio,
    CAST(nuevos.latitud AS DOUBLE),
    CAST(nuevos.longitud AS DOUBLE),
    CAST(nuevos.cantidad_paraderos AS DOUBLE), CAST(nuevos.distancia_min_m_paraderos AS DOUBLE),
    CAST(nuevos.cantidad_estaciones_metro AS DOUBLE), CAST(nuevos.distancia_min_m_estaciones_metro AS DOUBLE),
    CAST(nuevos.cantidad_jardines_infantiles AS DOUBLE), CAST(nuevos.distancia_min_m_jardines_infantiles AS DOUBLE),
    CAST(nuevos.cantidad_colegios AS DOUBLE), CAST(nuevos.distancia_min_m_colegios AS DOUBLE),
    CAST(nuevos.cantidad_universidades AS DOUBLE), CAST(nuevos.distancia_min_m_universidades AS DOUBLE),
    CAST(nuevos.cantidad_plazas AS DOUBLE), CAST(nuevos.distancia_min_m_plazas AS DOUBLE),
    CAST(nuevos.cantidad_supermercados AS DOUBLE), CAST(nuevos.distancia_min_m_supermercados AS DOUBLE),
    CAST(nuevos.cantidad_farmacias AS DOUBLE), CAST(nuevos.distancia_min_m_farmacias AS DOUBLE),
    CAST(nuevos.cantidad_centros_comerciales AS DOUBLE), CAST(nuevos.distancia_min_m_centros_comerciales AS DOUBLE),
    CAST(nuevos.cantidad_hospitales AS DOUBLE), CAST(nuevos.distancia_min_m_hospitales AS DOUBLE),
    CAST(nuevos.cantidad_clinicas AS DOUBLE), CAST(nuevos.distancia_min_m_clinicas AS DOUBLE),
    CAST(nuevos.fecha_scrapeo AS DATE)
)